In [1]:
import os
import json
import torch
import pickle
import numpy as np
import pandas as pd
import gdown
from tqdm import tqdm
import requests
import re
import torch.nn.functional as F
from collections import defaultdict
from torch_geometric.nn import SAGEConv
from sklearn.metrics.pairwise import cosine_similarity
from utils_optc.data_process_train import *
from utils_optc.data_process_test import *
from utils_optc.evaluate_darpatc import *

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


# Optc Specific Functions

In [2]:
def load_pkl(fname):
    with open(fname, 'rb') as f:
        obj = pickle.load(f)
    return obj

def load_events_from_hosts(hosts):
    all_events = []
    for host in hosts:
        path = f'../SysClient0{host}.systemia.com.txt'
        with open(path, 'r') as file:
            raw_events = [json.loads(line) for line in file]
        all_events.extend(raw_events)
    return all_events

def load_ground_truth(gt_file):
    with open(gt_file, 'r') as file:
        gt_nodes = set(file.read().split())
    return gt_nodes

In [3]:
# Functions to read the optc dataframe
def is_valid_entry(entry):
    valid_objects = {'PROCESS', 'FILE', 'FLOW', 'MODULE'}
    invalid_actions = {'START', 'TERMINATE'}

    object_valid = entry['object'] in valid_objects
    action_valid = entry['action'] not in invalid_actions
    actor_object_different = entry['actorID'] != entry['objectID']

    return object_valid and action_valid and actor_object_different

def Traversal_Rules(data):
    filtered_data = {}

    for entry in data:
        if is_valid_entry(entry):
            key = (
                entry['action'], 
                entry['actorID'], 
                entry['objectID'], 
                entry['object'], 
                entry['pid'], 
                entry['ppid']
            )
            filtered_data[key] = entry

    return list(filtered_data.values())

def Sentence_Construction(entry):
    action = entry["action"]
    properties = entry['properties']
    object_type = entry['object']

    format_strings = {
        'PROCESS': "{parent_image_path} {action} {image_path} {command_line}",
        'FILE': "{image_path} {action} {file_path}",
        'FLOW': "{image_path} {action} {src_ip} {src_port} {dest_ip} {dest_port} {direction}",
        'MODULE': "{image_path} {action} {module_path}"
    }

    default_format = "{image_path} {action} {module_path}"

    try:
        format_str = format_strings.get(object_type, default_format)
        phrase = format_str.format(action=action, **properties)
    except KeyError:
        phrase = ''

    return phrase.split(' ')

def Extract_Semantic_Info(event):
    object_type = event['object']
    properties = event['properties']

    label_mapping = {
        "PROCESS": ('parent_image_path', 'image_path'),
        "FILE": ('image_path', 'file_path'),
        "MODULE": ('image_path', 'module_path'),
        "FLOW": ('image_path', 'dest_ip', 'dest_port')
    }

    label_keys = label_mapping.get(object_type, None)
    if label_keys:
        labels = [properties.get(key) for key in label_keys]
        if all(labels):
            event["actorname"], event["objectname"] = labels[0], ' '.join(labels[1:])
            return event
    return None

def transform(text):
    labeled_data = [event for event in (Extract_Semantic_Info(x) for x in text) if event]
    data = Traversal_Rules(labeled_data)

    phrases = [Sentence_Construction(x) for x in data if Sentence_Construction(x)]
    for datum, phrase in zip(data, phrases):
        datum['phrase'] = phrase

    df = pd.DataFrame(data)
    s=0
    try:
        df['timestamp'] = pd.to_datetime(df['timestamp'].str[:-6], infer_datetime_format=True)
    except:
        s+=1
    df.sort_values(by='timestamp', inplace=True)

    return df

# Helpful functions

In [4]:
class SAGENet(torch.nn.Module):
	def __init__(self, in_channels, out_channels):
		super(SAGENet, self).__init__()	
		self.conv1 = SAGEConv(in_channels, 32, normalize=False)
		self.conv2 = SAGEConv(32, out_channels, normalize=False)

	def forward(self, x, edge_index):
		x = self.conv1(x, edge_index)
		x = F.relu(x)
		x = F.dropout(x, training=self.training)
		x = self.conv2(x, edge_index)
		return F.log_softmax(x, dim=1)

In [5]:
FOLDER_URL = "https://drive.google.com/drive/folders/1TR3yWxiAFrYc-0rw8dgMWHphEO__T0lc"

output_dir = os.path.abspath("../data/optc")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=True
)

Retrieving folder contents


Processing file 1wk-NPuEDjzLFnqMS1fQ2i4K89IwspFRV SysClient0051.systemia.com.txt
Processing file 1nvKPvgFYMa1zEe6ys-3xbqk84TTLHeJI SysClient0201.systemia.com.txt
Processing file 1L7kdMi9u3xzPkA2XmNYcbn66XrrXEKnw SysClient0501.systemia.com.txt


Retrieving folder contents completed
Building directory structure
Building directory structure completed


FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1wk-NPuEDjzLFnqMS1fQ2i4K89IwspFRV

but Gdown can't. Please check connections and permissions.

In [ ]:
FOLDER_URL = "https://drive.google.com/drive/folders/1YOfz1ULk0sIy_reiFlkJRPyklFDvQNnc"

output_dir = os.path.abspath("../groundtruth/optc_plain_powershell")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=True
)

In [ ]:
os.makedirs("optc_resources", exist_ok=True)

file_ids = [
    "1Wtul5b5sb3ERL5hhkeGgvwwoGiBZaiLO",
    "1RdWLNtuAz3QbRVgAhD3hKcDjoJPDeK3t",
]

def get_filename(file_id):
    url = f"https://drive.google.com/file/d/{file_id}/view"
    html = requests.get(url).text
    match = re.search(r'<meta property="og:title" content="(.+?)"', html)
    return match.group(1) if match else file_id

for fid in file_ids:
    filename = get_filename(fid)
    out_path = os.path.join("optc_resources", filename)
    url = f"https://drive.google.com/uc?id={fid}"
    
    print(f"⬇️ Downloading: {filename}")
    gdown.download(url, out_path, quiet=False)

## Download the models and features

In [ ]:
FOLDER_URL = "https://drive.google.com/drive/folders/17ZJMuGA8KtLK6iQWEx-aHi2ocXwLYC28"

output_dir = os.path.abspath("../models/optc_plain_powershell")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=False
)

print(f"All files downloaded into: {output_dir}")

In [ ]:
FOLDER_URL = "https://drive.google.com/drive/folders/1PXdW7XjT3sVQ69DHWXBWtj_aNHxbchi2"

output_dir = os.path.abspath("../models/optc_plain_powershell")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=False
)

print(f"All files downloaded into: {output_dir}")

In [ ]:
FOLDER_URL = "https://drive.google.com/drive/folders/1Lsvye-uBsrF6ORgCtiUNux0-h6RaGuW4"

output_dir = os.path.abspath("../groundtruth/optc_plain_powershell")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=False
)

print(f"All files downloaded into: {output_dir}")

In [ ]:
FOLDER_URL = "https://drive.google.com/drive/folders/1YOfz1ULk0sIy_reiFlkJRPyklFDvQNnc"

output_dir = os.path.abspath("../groundtruth/optc_plain_powershell")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=True
)

# Load Dataset

In [ ]:
all_events = load_events_from_hosts(['201'])
df_plain_attack = transform(all_events)

EnActIds = [x['actorID'] for x in all_events]
EnObjIds = [x['objectID'] for x in all_events]
EntitySet = set(EnActIds).union(set(EnObjIds))

gt_nodes = load_ground_truth('./optc_resources/optc.txt')
gt_nodes = [x for x in gt_nodes if x in EntitySet]
gt_nodes = set(gt_nodes)

print("Number of malicious nodes:", len(gt_nodes))


In [ ]:
# Drop the unnecessary columns
columns_to_remove = ['hostname', 'id', 'pid', 'ppid', 'principal', 'properties', 'phrase', 'tid']
df_plain_attack_modified = df_plain_attack.drop(columns=columns_to_remove, errors='ignore')

# Add actor_type column with constant value "PROCESS"
df_plain_attack_modified['actor_type'] = 'PROCESS'

# Rename actorname → exec and objectname → path
df_plain_attack_modified = df_plain_attack_modified.rename(columns={
    'actorname': 'exec',
    'objectname': 'path'
})

# Reorder columns
final_columns = ['actorID', 'actor_type', 'objectID', 'object', 'action', 'timestamp', 'exec', 'path']
df_plain_attack_modified = df_plain_attack_modified[final_columns]

df_plain_attack_modified.head()


### Load test data

In [ ]:
# --- Load test dataset ---
path = 'optc_resources/optc_plain_powershell_test.txt' 
data1, feature_num, label_num, adj, adj2, nodeA = MyDatasetA(path, 'optc_plain_powershell')
dataset_test = TestDatasetA(data1)
data_test = dataset_test[0] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def test_one_model(mask, model, device, data, thre, nodeA):
    model.eval()
    data = data.to(device)

    # start with all nodes flagged as suspicious
    updated_mask = torch.ones(data.num_nodes, dtype=torch.bool, device=device)


    # Convert malicious list to set for O(1) lookup
    malicious_set = set(nodeA)

    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        pro = torch.softmax(out, dim=1)
        pro1 = pro.max(1)

        # suppress top prediction to find 2nd-best for thresholding
        pro_clone = pro.clone()
        pro_clone[torch.arange(len(pro)), pro1[1]] = -1
        pro2 = pro_clone.max(1)

        # apply threshold logic
        uncertain_mask = (pro1[0] / pro2[0]) < thre
        print("Number of uncertain predictions:", uncertain_mask.sum().item())
        pred[uncertain_mask] = 100  # dummy label for uncertain predictions

        # evaluation
        TP, FP, TN, FN = 0, 0, 0, 0
        test_nodes = mask.nonzero(as_tuple=False).view(-1).tolist()

        for i in tqdm(test_nodes, desc="Evaluating nodes"):
            flagged = (pred[i] != data.y[i])  # mismatch = flagged
            
            if not flagged:
                updated_mask[i] = False   # node is cleared

            if i in malicious_set:  # malicious node
                if flagged:
                    TP += 1
                else:
                    FN += 1
            else:  # benign node
                if flagged:
                    FP += 1
                else:
                    TN += 1

        # metrics
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0

        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()

    return acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask


In [ ]:
# acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask = test_one_model(data_test.test_mask, initial_model, device, data_test, thre, nodeA)

# print(f"Test Accuracy: {acc:.4f}")
# print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
# print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")

# Evaluating the model with best FP TN ratio

In [ ]:
model_good = SAGENet(feature_num, label_num).to(device)

# Load saved weights
load_path = "../models/optc_plain_powershell/optc_plain_powershell.pt"
model_good.load_state_dict(torch.load(load_path, map_location=device))

acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask = test_one_model(data_test.test_mask, model_good, device, data_test, thre, nodeA)

print(f"Test Accuracy: {acc:.4f}")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")

In [ ]:
num_true = int(updated_mask.sum().item())   # number of True entries
num_false = int((~updated_mask).sum().item())  # number of False entries
print("True:", num_true, "False:", num_false)


In [ ]:
def evaluate(dataset_name):

    eps = 1e-10   # small epsilon to avoid division by zero when computing metrics

    # Open the ground truth malicious node IDs
    f_gt = open(f'../groundtruth/{dataset_name}/groundtruth_nodeId.txt', 'r')
    # Open the alarm file (contains detection results)
    f_alarm = open(f'../groundtruth/{dataset_name}/alarm.txt', 'r')

    # Dictionary of ground truth malicious nodes
    gt = {}
    for line in f_gt:
        gt[int(line.strip('\n').split(' ')[0])] = 1  # mark each malicious node ID as 1

    # List that will hold classification results for each node ("tp", "fp", "tn", "fn")
    ans = []

    # Process each line of the alarm file
    for line in f_alarm:
        if line == '\n': 
            continue   # skip empty lines

        if not ':' in line:
            # If the line has no ":", it specifies the total number of nodes
            tot_node = int(line.strip('\n'))
            # Initialize all nodes as "tn" (true negative)
            for i in range(tot_node):
                ans.append('tn')
            # Then overwrite malicious nodes as "fn" (false negative) initially
            for i in gt:
                ans[i] = 'fn'
            continue

        # Otherwise, the line describes an alarm
        line = line.strip('\n')
        a = int(line.split(':')[0])                  # the node that raised the alarm
        b = line.split(':')[1].strip(' ').split(' ') # list of related node IDs

        flag = 0
        # Check each related node in b
        for i in b:
            if i == '': 
                continue
            if int(i) in gt.keys():          # if related node is malicious
                ans[int(i)] = 'tp'           # mark as true positive
                flag = 1                     # detection was correct

        # Now check the main node a
        if a in gt.keys():
            ans[a] = 'tp'                    # if alarmed node is malicious → true positive
        else:
            if flag == 0:                    # if none of the related nodes were malicious
                ans[a] = 'fp'                # alarmed node is benign → false positive

    # Count metrics
    tn = 0
    tp = 0
    fn = 0
    fp = 0
    for i in ans:
        if i == 'tp': tp += 1
        if i == 'tn': tn += 1
        if i == 'fp': fp += 1
        if i == 'fn': fn += 1

    # Print confusion matrix values
    print("TP:", tp, "FP:", fp, "TN:", tn, "FN:", fn)

    # Compute metrics with epsilon to prevent division by zero
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    fscore = 2 * precision * recall / (precision + recall + eps)

    # Print metrics
    print('Precision: ', precision)
    print('Recall: ', recall)
    print('F-Score: ', fscore)


# Run evaluation on dataset "optc_plain_powershell"
evaluate('optc_plain_powershell')


# Evasion

In [ ]:
def add_node_properties(nodes, node_id, properties):
    if node_id not in nodes:
        nodes[node_id] = []
    nodes[node_id].extend(properties)

def update_edge_index(edges, edge_index, index):
    for src_id, dst_id in edges:
        src = index[src_id]
        dst = index[dst_id]
        edge_index[0].append(src)
        edge_index[1].append(dst)

def prepare_graph(df):
    nodes, labels, edges = {}, {}, []
    dummies = {"PROCESS":0, "MODULE":1, "FILE": 2, "FLOW":3}
    
    for _, row in df.iterrows():
        action = row["action"]
        properties = [row['exec'], action] + ([row['path']] if row['path'] else [])
        
        actor_id = row["actorID"]
        add_node_properties(nodes, actor_id, properties)
        labels[actor_id] = dummies[row['actor_type']]

        object_id = row["objectID"]
        add_node_properties(nodes, object_id, properties)
        labels[object_id] = dummies[row['object']]

        edges.append((actor_id, object_id))

    features, feat_labels, edge_index, index_map = [], [], [[], []], {}
    for node_id, props in nodes.items():
        features.append(props)
        feat_labels.append(labels[node_id])
        index_map[node_id] = len(features) - 1

    update_edge_index(edges, edge_index, index_map)

    return features, feat_labels, edge_index, list(index_map.keys())

phrases, labels, edges, mapp = prepare_graph(df_plain_attack_modified)

# Step 1: Benign Node Selection

In [ ]:
def split_malicious_nodes_by_label(malicious_indices, labels):

    r"""Groups malicious nodes based on their labels.

    :param malicious_indices: List of indices for malicious nodes.
    :param labels: List of labels for all nodes.
    :return: Dictionary with label as key and list of malicious node indices as values.
    """

    malicious_groups = defaultdict(list)

    for node_idx in malicious_indices:
        node_label = labels[node_idx]
        malicious_groups[node_label].append(node_idx)

    return dict(malicious_groups)

malicious_groups = split_malicious_nodes_by_label(nodeA, labels)

print("The number of malicious nodes for each label is as follows:")

for label, nodes in malicious_groups.items():
    print(f"Label {label}: {len(nodes)}")  

In [ ]:
def split_nodes_by_label(nodes, labels):

    r"""Groups all nodes based on their labels.

    :param nodes: List of phrases (interactions) for all nodes.
    :param labels: List of labels for all nodes.
    :return: Dictionary with label as key and list of node indices as values.
    """

    node_groups = defaultdict(list)
    for node_idx, _ in enumerate(nodes):
        node_label = int(labels[node_idx].item()) 
        node_groups[node_label].append(node_idx)
    return dict(node_groups)

# Example call
all_nodes_grouped = split_nodes_by_label(data_test.x, data_test.y)

print("The number of nodes (either malicious or benign) for each label is as follows:")
for label, nodes in sorted(all_nodes_grouped.items()):
    print(f"Label {label}: {len(nodes)}")


In [ ]:
def split_nodes_by_label_exclude_malicious(nodes, labels, indices_of_malicious_nodes):

    r"""Groups benign (non-malicious) nodes based on their labels.

    :param nodes: List of phrases (interactions) for all nodes.
    :param labels: List of labels for all nodes.
    :param indices_of_malicious_nodes: List of indices for malicious nodes.
    :return: Dictionary with label as key and list of benign node indices as values.
    """

    node_groups = defaultdict(list)
    malicious_set = set(indices_of_malicious_nodes)  

    for node_idx, _ in enumerate(nodes):
        if node_idx not in malicious_set:
            node_label = int(labels[node_idx].item())  # ensure int key
            node_groups[node_label].append(node_idx)

    return dict(node_groups)


# Example call
all_nodes_grouped_excluding_malicious = split_nodes_by_label_exclude_malicious(
    data_test.x, data_test.y, nodeA
)

print("The number of candidate benign nodes for each label is as follows:")
for label, nodes in sorted(all_nodes_grouped_excluding_malicious.items()):
    print(f"Label {label}: {len(nodes)}")


# Step 2: Minimal Interaction Selection

In [ ]:
def filter_nodes_by_phrase_length(node_groups, phrases, min_len=5, max_len=15):

    r"""Filters the benign nodes in each label based on the number of interactions (phrase length).

    :param node_groups: Dictionary with label as key and list of benign node indices as values.
    :param phrases: List of phrases (interactions) for all nodes.
    :param min_len: Minimum number of interactions (hyperparameter).
    :param max_len: Maximum number of interactions (hyperparameter).
    :return: Dictionary with label as key and list of filtered benign node indices as values.
    """

    filtered_groups = {}

    for label, node_indices in node_groups.items():
        # Apply filtering
        filtered = [idx for idx in node_indices if min_len <= len(phrases[idx]) <= max_len]
        
        # If filtering removes everything, keep original nodes
        if len(filtered) == 0:
            filtered_groups[label] = node_indices
        else:
            filtered_groups[label] = filtered

    return filtered_groups


filtered_nodes_grouped = filter_nodes_by_phrase_length(
    all_nodes_grouped_excluding_malicious, phrases
)

print("The number of candidate benign nodes for each label is as follows:")
for label, nodes in filtered_nodes_grouped.items():
    print(f"Label {label}: {len(nodes)}")


# Step 3: Contextual Consistency Filtering

In [ ]:
def compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes):

    r"""Orders the benign nodes within each label based on their contextual similarity to the malicious nodes in the same label in descending order.

    :param malicious_groups: Dictionary with label as key and list of malicious node indices as values.
    :param filtered_nodes_grouped: Dictionary with label as key and list of filtered benign node indices as values.
    :param nodes: Feature vectors (embeddings) for all nodes.
    :return: Dictionary with label as key and list of sorted benign node indices as values.
    """

    top_nodes_by_similarity = {}

    for label in tqdm(malicious_groups.keys()):
        if label in filtered_nodes_grouped:
            malicious_indices = malicious_groups[label]
            filtered_indices = filtered_nodes_grouped[label]

            # Get feature vectors for malicious and filtered nodes
            malicious_vectors = np.array([nodes[idx] for idx in malicious_indices])
            filtered_vectors = np.array([nodes[idx] for idx in filtered_indices])

            # Compute cosine similarity (malicious nodes vs filtered nodes)
            similarities = cosine_similarity(filtered_vectors, malicious_vectors)  # Shape: (filtered, malicious)

            # Take the max similarity score for each filtered node
            max_similarities = similarities.max(axis=1)  # Max similarity per filtered node

            # Use np.argsort to get the indices that would sort the max_similarities array
            sorted_indices = np.array(filtered_indices)[np.argsort(-max_similarities)]  # Negative for descending order

            # Select top most similar nodes for the current label
            top_nodes_by_similarity[label] = sorted_indices

    return top_nodes_by_similarity

top_nodes_by_similarity = compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, data_test.x.cpu().numpy() )

print("The number of candidate benign nodes for each label is as follows:")

for label, top_nodes in top_nodes_by_similarity.items():
    print(f"Label {label}: {len(top_nodes)}") 

print(" ")

# Graph Structure Adjustment

In [ ]:
# Prepare the DataFrame for modification
print("Number of rows in df before modification:", len(df_test))

malicious_to_benign = {}
for i in nodeA:
    if labels[i] in top_nodes_by_similarity and len(top_nodes_by_similarity[labels[i]]) > 0:
        malicious_to_benign[mapp[i]] = mapp[top_nodes_by_similarity[labels[i]][0]]

# Step 2: Retrieve all benign node IDs that will be replaced
all_benign_replacements = set(malicious_to_benign.values())

# Step 3: Extract relevant rows for each benign ID
benign_rows_dict = {benign_id: df_plain_attack_modified[(df_plain_attack_modified['actorID'] == benign_id) | (df_plain_attack_modified['objectID'] == benign_id)].copy()
                    for benign_id in all_benign_replacements}

new_rows = []  # Store newly modified rows

for malicious_id, benign_id in tqdm(malicious_to_benign.items(), desc="Processing malicious nodes"):
    if benign_id in benign_rows_dict:
        modified_rows = benign_rows_dict[benign_id].copy()  # Copy rows where benign ID appears
        # If the benign node is in actorID, insert the malicious node in objectID
        modified_rows.loc[modified_rows['actorID'] == benign_id, 'actorID'] = malicious_id
        # If the benign node is in objectID, insert the malicious node in actorID
        modified_rows.loc[modified_rows['objectID'] == benign_id, 'objectID'] = malicious_id
        # Append modified rows to the new list
        new_rows.append(modified_rows)

# Step 5: Concatenate all new rows once for efficiency
df_curated = pd.concat([df_test] + new_rows, ignore_index=True)

print("Number of rows in df after modification:", len(df_curated))
print("The number of triggered events are: ", len(df_curated) - len(df_test))


In [ ]:
# Save the curated DataFrame to a text file (tab-separated, no header, no index)
output_path = "optc_resources/optc_plain_powershell_test_curated.txt"
df_curated.to_csv(output_path, sep='\t', index=False, header=False)

print(f"Curated file saved as {output_path}")


# Eval

In [ ]:
# --- Load test dataset ---
path = 'optc_resources/optc_plain_powershell_test_curated.txt' 
data_curated, feature_num, label_num, adj, adj2, nodeA = MyDatasetA(path, 'optc_plain_powershell')
dataset_curated = TestDatasetA(data_curated)
data_curated = dataset_curated[0] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
load_path = "../models/optc_plain_powershell/optc_plain_powershell.pt"
model_good.load_state_dict(torch.load(load_path, map_location=device))

# Put in eval mode for inference
model_good.eval()

print("Model loaded successfully:", type(model_good))

acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask_evasion = test_one_model(data_curated.test_mask, model_good, device, data_curated, thre, nodeA)

print(f"Test Accuracy: {acc:.4f}")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")